In [3]:
!pip uninstall -y lion-linker

Found existing installation: lion_linker 0.1.0
Uninstalling lion_linker-0.1.0:
  Successfully uninstalled lion_linker-0.1.0


In [6]:
! pip install lion-linker


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [7]:
from lion_linker.retrievers import LamapiClient
import os
client = LamapiClient(
    endpoint=os.getenv("ENTITY_RETRIEVAL_ENDPOINT"),
    token=os.getenv("ENTITY_RETRIEVAL_TOKEN"),
    num_candidates=10
)
await client.fetch_multiple_entities(["paris"])

{'paris': [{'id': 'Q42291481',
   'name': 'Paris Paris',
   'description': '2018 Tamil film directed by Ramesh Aravind',
   'types': [{'id': 'Q11424', 'name': 'film'}],
   'kind': 'entity',
   'NERtype': 'OTHERS',
   'ambiguity_mention': 0.5,
   'corrects_tokens': 1.0,
   'ntoken_mention': 1,
   'ntoken_entity': 2,
   'length_mention': 5,
   'length_entity': 11,
   'popularity': 0.0,
   'pos_score': 0.1,
   'es_score': 1.0,
   'ed_score': 0.45,
   'jaccard_score': 1.0,
   'jaccardNgram_score': 1.0},
  {'id': 'Q111210960',
   'name': 'Paris Paris',
   'description': 'Canadian television series',
   'types': [{'id': 'Q5398426', 'name': 'television series'}],
   'kind': 'entity',
   'NERtype': 'OTHERS',
   'ambiguity_mention': 0.5,
   'corrects_tokens': 1.0,
   'ntoken_mention': 1,
   'ntoken_entity': 2,
   'length_mention': 5,
   'length_entity': 11,
   'popularity': 0.0,
   'pos_score': 0.2,
   'es_score': 1.0,
   'ed_score': 0.45,
   'jaccard_score': 1.0,
   'jaccardNgram_score': 1.0},

In [ ]:
import os
from lion_linker.lion_linker import LionLinker
from lion_linker.retrievers import OpenRefineClient
import pandas as pd

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENROUTER_API_KEY for OpenRouter")

# Simple hardcoded table with headers
df = pd.DataFrame(
    [
        {"Title": "The Matrix", "Year": 1999, "Rating": 8.7, "Genre": "Sci-Fi", "Votes": 463517, "Director": "Lana Wachowski"},
        {"Title": "Inception", "Year": 2010, "Rating": 8.8, "Genre": "Sci-Fi", "Votes": 2000000, "Director": "Christopher Nolan"},
        {"Title": "Parasite", "Year": 2019, "Rating": 8.5, "Genre": "Thriller", "Votes": 800000, "Director": "Bong Joon-ho"},
    ]
)
df.to_csv("mini_movies.csv", index=False)

# Simple retriever (no token needed)
retriever = OpenRefineClient(num_candidates=5)

linker = LionLinker(
    input_csv="mini_movies.csv",
    model_name="gpt-oss-120b",  # OpenRouter model alias
    retriever=retriever,
    output_csv="mini_movies_output.csv",
    mention_columns=["Title"],
    model_api_provider="openrouter",
    model_api_key=api_key,
    table_ctx_size=0,
    chunk_size=2,
)

await linker.run()
print("Wrote:", linker.output_csv)


2026-01-12 10:22:17,394 - INFO - Model API provider is: openrouter
2026-01-12 10:22:17,403 - INFO - Initializing components...
2026-01-12 10:22:17,404 - INFO - Knowledge graph: generic
2026-01-12 10:22:17,404 - INFO - Setup completed.
2026-01-12 10:22:17,405 - INFO - Starting processing of mini_movies.csv...
2026-01-12 10:22:17,812 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-12 10:22:23,209 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-12 10:22:23,719 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-12 10:22:25,489 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
Processing Batches: 100%|██████████| 3/3 [00:08<00:00,  2.72s/it]
2026-01-12 10:22:25,567 - INFO - Processing completed. Results are incrementally saved to mini_movies_output.csv


Wrote: mini_movies_output.csv


In [1]:
! pip uninstall -y crocodile-linker

In [2]:
! pip install crocodile-linker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.0/928.0 kB 15.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [3]:
from crocodile import Crocodile
import pandas as pd
import os

# Simple hardcoded table with headers
df = pd.DataFrame(
    [
        {"Title": "The Matrix", "Year": 1999, "Rating": 8.7, "Genre": "Sci-Fi", "Votes": 463517, "Director": "Lana Wachowski"},
        {"Title": "Inception", "Year": 2010, "Rating": 8.8, "Genre": "Sci-Fi", "Votes": 2000000, "Director": "Christopher Nolan"},
        {"Title": "Parasite", "Year": 2019, "Rating": 8.5, "Genre": "Thriller", "Votes": 800000, "Director": "Bong Joon-ho"},
    ]
)

crocodile_instance = Crocodile(
    input_csv=df,
    table_name="imdb",
    dataset_name="cinema",
    max_candidates=3,
    entity_retrieval_token=os.environ["ENTITY_RETRIEVAL_TOKEN"],
    entity_retrieval_endpoint=os.environ["ENTITY_RETRIEVAL_ENDPOINT"],
    entity_bow_endpoint=os.environ["ENTITY_BOW_ENDPOINT"],
    columns_type={
        "NE": {"0": "OTHER"},
        "LIT": {
            "1": "NUMBER",
            "2": "NUMBER",
            "3": "STRING",
            "4": "NUMBER",
            "5": "STRING",
        },
        "IGNORED": [],
    },
    output_csv="./crocodile_output.csv",
    mongo_uri="mongodb://localhost:27017/",  # ← correct
)

# Run the entity linking process
crocodile_instance.run()

print("Entity linking process completed.")


Onboarding 3 rows for dataset 'cinema', table 'imdb'
Chunk 1/1: Onboarded rows 1-3 (573.5 rows/sec)
Data onboarding complete for dataset 'cinema' and table 'imdb'
Onboarded 3 rows in 0.01 seconds (570.8 rows/sec)
Found 3 tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
No more tasks to process.
Computing type-frequency features by randomly sampling 2 documents
Computed type frequencies from 2 documents
ML ranking progress: 0/3 documents
ML ranking progress: 0/3 documents
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
ML ranking progress: 1/3 documents
ML ranking progress: 2/3 documents
ML ranking complete: 1/3 documents
ML ranking complete: 2/3 documents
All tasks have been processed.
Processing 3 documents

In [3]:
! pip install alligator-linker

  Using cached alligator_linker-0.1.0-py3-none-any.whl.metadata (11 kB)
Using cached alligator_linker-0.1.0-py3-none-any.whl (2.0 MB)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
! pip uninstall -y alligator-linker

Found existing installation: alligator-linker 0.1.0
Uninstalling alligator-linker-0.1.0:
  Successfully uninstalled alligator-linker-0.1.0


In [4]:
from alligator import Alligator
import pandas as pd
import os

# Simple hardcoded table with headers
df = pd.DataFrame(
    [
        {"Title": "The Matrix", "Year": 1999, "Rating": 8.7, "Genre": "Sci-Fi", "Votes": 463517, "Director": "Lana Wachowski"},
        {"Title": "Inception", "Year": 2010, "Rating": 8.8, "Genre": "Sci-Fi", "Votes": 2000000, "Director": "Christopher Nolan"},
        {"Title": "Parasite", "Year": 2019, "Rating": 8.5, "Genre": "Thriller", "Votes": 800000, "Director": "Bong Joon-ho"},
    ]
)

alligator_instance = Alligator(
    input_csv=df,
    table_name="imdb",
    dataset_name="cinema",
    max_candidates_in_result=3,
    candidate_retrieval_limit=10,
    entity_retrieval_token=os.environ["ENTITY_RETRIEVAL_TOKEN"],
    entity_retrieval_endpoint=os.environ["ENTITY_RETRIEVAL_ENDPOINT"],
    object_retrieval_endpoint=os.environ.get("OBJECT_RETRIEVAL_ENDPOINT"),
    literal_retrieval_endpoint=os.environ.get("LITERAL_RETRIEVAL_ENDPOINT"),
    target_columns={
        "NE": {"0": "OTHER"},
        "LIT": {
            "1": "NUMBER",
            "2": "NUMBER",
            "3": "STRING",
            "4": "NUMBER",
            "5": "STRING",
        },
        "IGNORED": [],
    },
    output_csv="./alligator_output.csv",
    mongo_uri="mongodb://localhost:27017/",
)

# Run the entity linking process
alligator_instance.run()

print("Entity linking process completed.")


[2026-01-12 10:17:24 - alligator.coordinator - INFO] Starting Alligator entity linking pipeline...
[2026-01-12 10:17:24 - alligator.coordinator - INFO] Step 1: Data onboarding...
[2026-01-12 10:17:24 - alligator.data_manager - INFO] Onboarding 3 rows for dataset 'cinema', table 'imdb'
[2026-01-12 10:17:24 - alligator.coordinator - INFO] Data onboarding complete - 3 rows in 0.28s
[2026-01-12 10:17:24 - alligator.coordinator - INFO] Step 2: Running workers for candidate retrieval and processing...
[2026-01-12 10:17:24 - alligator.worker_manager - INFO] Found 3 tasks to process.
[2026-01-12 10:17:27 - alligator.worker_manager - INFO] Worker 0 started.
[2026-01-12 10:17:27 - alligator.worker_manager - INFO] Created HTTP session with limit 32 and SSL verify False.
[2026-01-12 10:17:28 - alligator.worker_manager - INFO] Worker 0 started with batch size 3, skipping 0.
[2026-01-12 10:17:28 - alligator.worker_manager - INFO] Worker 0 processing final batch of 3 documents.
[2026-01-12 10:17:28 -